# 🚀 LLM Inference Optimization Benchmark Suite

## Comparing Standard Multi-Head Attention, Ring Attention, and Rotary Positional Embedding (RoPE)

---

##  Overview

Large Language Models (LLMs) require significant computational resources during inference, particularly due to the self-attention mechanism. As model sizes and context lengths increase, attention becomes one of the primary bottlenecks in terms of memory consumption, computational cost, and inference latency.

To address these challenges, several optimization techniques have been proposed. Some methods reduce the memory required for Key-Value (KV) caches, others accelerate the attention computation itself, while some improve positional encoding for better long-context understanding.


---

#  Objectives

This Lab demonstrates:


- Ring Attention
- Rotary Positional Embedding (RoPE)

# Ring Attention

In [ ]:
# ============================================================
# Ring Attention
# ============================================================

total_tokens = 1_000_000

gpus = 4

tokens_per_gpu = total_tokens // gpus

print("="*60)
print("RING ATTENTION")
print("="*60)

for gpu in range(gpus):

    start = gpu*tokens_per_gpu + 1
    end = (gpu+1)*tokens_per_gpu

    print(f"GPU {gpu+1}")
    print(f"Processes Tokens {start:,} - {end:,}")
    print("Sends Information To Next GPU")
    print("-"*40)

print("\nAll GPUs cooperate to process the full context.")

RING ATTENTION
GPU 1
Processes Tokens 1 - 250,000
Sends Information To Next GPU
----------------------------------------
GPU 2
Processes Tokens 250,001 - 500,000
Sends Information To Next GPU
----------------------------------------
GPU 3
Processes Tokens 500,001 - 750,000
Sends Information To Next GPU
----------------------------------------
GPU 4
Processes Tokens 750,001 - 1,000,000
Sends Information To Next GPU
----------------------------------------

All GPUs cooperate to process the full context.


**Understanding from Scratch**

In [ ]:
# ============================================================
# Ring Attention Simulation
# ============================================================

import torch

num_gpus = 4

seq_len = 1024

chunk = seq_len // num_gpus

tokens = torch.arange(seq_len)

print("="*60)
print("RING ATTENTION")
print("="*60)

for gpu in range(num_gpus):

    local_tokens = tokens[gpu*chunk:(gpu+1)*chunk]

    print(f"GPU {gpu+1}")

    print("Local Chunk Shape :", local_tokens.shape)

    print("Computes Local Attention")

    print("Passes KV To Next GPU")

    print("-"*40)

print("Final Attention Computed Across All GPUs")

RING ATTENTION
GPU 1
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
GPU 2
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
GPU 3
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
GPU 4
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
Final Attention Computed Across All GPUs


**Linkage**

In [ ]:
print("""
                 GPU1
                  │
                  ▼
GPU4 ◄──────── GPU2
 │               │
 ▼               ▼
──────────── GPU3
""")


                 GPU1
                  │
                  ▼
GPU4 ◄──────── GPU2
 │               │
 ▼               ▼
──────────── GPU3



**Comparison**

In [ ]:
print("="*135)
print("FINAL COMPARISON")
print("="*135)

print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<20}".format(
    "Metric",
    "Standard",
    "MQA",
    "GQA",
    "Flash",
    "Ring"))

print("-"*135)

rows = [
("Keys","8","1","4","8","8"),
("Values","8","1","4","8","8"),
("Parameters","1,050,624","591,168","722,240","1,050,624","1,050,624"),
("KV Cache","Large","Smallest","Medium","Large","Distributed"),
("Attention Matrix","Stored","Stored","Stored","Not Stored","Distributed"),
("GPU Memory","Highest","Lowest","Medium","Very Low","Distributed"),
("Inference","Slow","Fastest","Fast","Very Fast","Scales to Multi-GPU"),
("Architecture","Original","Modified","Modified","Same","Same"),
("Single GPU","Yes","Yes","Yes","Yes","No"),
("Multi GPU","Optional","Optional","Optional","Optional","Required"),
("Context Length","Limited","Limited","Limited","Limited","Millions of Tokens"),
("Quality","Best","Slight Drop","Near Standard","Same","Same")
]

for r in rows:
    print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<20}".format(*r))

print("="*135)

FINAL COMPARISON
Metric            Standard      MQA           GQA           Flash         Ring                
---------------------------------------------------------------------------------------------------------------------------------------
Keys              8             1             4             8             8                   
Values            8             1             4             8             8                   
Parameters        1,050,624     591,168       722,240       1,050,624     1,050,624           
KV Cache          Large         Smallest      Medium        Large         Distributed         
Attention Matrix  Stored        Stored        Stored        Not Stored    Distributed         
GPU Memory        Highest       Lowest        Medium        Very Low      Distributed         
Inference         Slow          Fastest       Fast          Very Fast     Scales to Multi-GPU 
Architecture      Original      Modified      Modified      Same          Same         

**Rotary Positional Encoding**

**Probelm without PE**

In [ ]:
# ============================================================
# Without Positional Encoding
# ============================================================

sentence = ["The","Cat","Chased","The","Dog"]

print("="*60)
print("WITHOUT POSITIONAL INFORMATION")
print("="*60)

for word in sentence:

    print(word)

WITHOUT POSITIONAL INFORMATION
The
Cat
Chased
The
Dog


**Traditional PE**

In [ ]:
# ============================================================
# Traditional Positional Encoding
# ============================================================

sentence = ["The","Cat","Chased","The","Dog"]

print("="*60)
print("POSITION IDs")
print("="*60)

for position,word in enumerate(sentence):

    print(f"{word:10} Position = {position}")

POSITION IDs
The        Position = 0
Cat        Position = 1
Chased     Position = 2
The        Position = 3
Dog        Position = 4


**Rotart Positional Encoding**

In [ ]:
# ============================================================
# RoPE Simulation
# ============================================================

import numpy as np

positions = [0,1,2,3,4]

print("="*60)
print("ROTARY POSITIONAL EMBEDDING")
print("="*60)

for p in positions:

    angle = p*15

    print(f"Token Position {p}")
    print(f"Rotation Angle : {angle}°")
    print("-"*30)

ROTARY POSITIONAL EMBEDDING
Token Position 0
Rotation Angle : 0°
------------------------------
Token Position 1
Rotation Angle : 15°
------------------------------
Token Position 2
Rotation Angle : 30°
------------------------------
Token Position 3
Rotation Angle : 45°
------------------------------
Token Position 4
Rotation Angle : 60°
------------------------------


**Understanding from Scratch**

In [ ]:
import torch

def rotate_half(x):

    x1 = x[...,::2]

    x2 = x[...,1::2]

    return torch.cat((-x2,x1),dim=-1)


def apply_rope(x):

    B,H,T,D = x.shape

    position = torch.arange(T).float()

    freq = torch.arange(0,D,2).float()/D

    theta = 10000**(-freq)

    angles = position[:,None]*theta[None,:]

    sin = torch.sin(angles)

    cos = torch.cos(angles)

    sin = sin.repeat_interleave(2,-1)

    cos = cos.repeat_interleave(2,-1)

    sin = sin.unsqueeze(0).unsqueeze(0)

    cos = cos.unsqueeze(0).unsqueeze(0)

    return x*cos + rotate_half(x)*sin

In [ ]:
Q = torch.randn(2,8,128,64)

Q_rope = apply_rope(Q)

print("="*60)
print("ROTARY POSITIONAL EMBEDDING")
print("="*60)

print("Original Shape :",Q.shape)

print("RoPE Shape     :",Q_rope.shape)

print()

print("Original Vector")

print(Q[0,0,1,:8])

print()

print("After Rotation")

print(Q_rope[0,0,1,:8])

ROTARY POSITIONAL EMBEDDING
Original Shape : torch.Size([2, 8, 128, 64])
RoPE Shape     : torch.Size([2, 8, 128, 64])

Original Vector
tensor([ 0.3223,  2.0287, -1.5426,  2.2404,  0.1623,  0.2967,  1.8046,  0.7467])

After Rotation
tensor([-1.5329, -0.7892, -1.3310,  1.1305,  0.4428,  0.1130,  2.1055,  0.6674])


In [ ]:
import torch
import torch.nn as nn

# Dummy Input
B = 2
T = 128
D = 512
H = 8
head_dim = D // H

x = torch.randn(B, T, D)

# Projection Layers
Wq = nn.Linear(D, D)
Wk = nn.Linear(D, D)
Wv = nn.Linear(D, D)

# Project
Q = Wq(x)
K = Wk(x)
V = Wv(x)

# Split into heads
Q = Q.view(B, T, H, head_dim).transpose(1, 2)
K = K.view(B, T, H, head_dim).transpose(1, 2)
V = V.view(B, T, H, head_dim).transpose(1, 2)

# Apply RoPE
Q = apply_rope(Q)
K = apply_rope(K)

# Compute Attention Scores
scores = (Q @ K.transpose(-2, -1)) / (head_dim ** 0.5)

print("Q Shape :", Q.shape)
print("K Shape :", K.shape)
print("V Shape :", V.shape)
print("Attention Scores Shape :", scores.shape)

Q Shape : torch.Size([2, 8, 128, 64])
K Shape : torch.Size([2, 8, 128, 64])
V Shape : torch.Size([2, 8, 128, 64])
Attention Scores Shape : torch.Size([2, 8, 128, 128])


# Complete Comparison

In [ ]:
print("="*150)
print("LLM INFERENCE OPTIMIZATION COMPARISON")
print("="*150)

print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<14}{:<22}".format(
    "Metric",
    "Standard",
    "MQA",
    "GQA",
    "Flash",
    "Ring",
    "RoPE"))

print("-"*150)

rows = [
("Parameters","1.05M","0.59M","0.72M","1.05M","1.05M","1.05M"),
("KV Cache","Large","Small","Medium","Large","Distributed","Large"),
("Attention Matrix","Stored","Stored","Stored","Not Stored","Distributed","Stored"),
("Memory","Highest","Lowest","Medium","Very Low","Distributed","Same"),
("Inference","Slow","Fastest","Fast","Very Fast","Scalable","Same"),
("Context Length","Limited","Limited","Limited","Limited","Millions","Much Longer"),
("Architecture","Original","Modified","Modified","Same","Same","Same"),
("Changes","None","Share KV","Group KV","Kernel","Distributed","Rotate Q/K"),
("Quality","Baseline","Slight Drop","Near Baseline","Same","Same","Better Long Context")
]

for r in rows:
    print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<14}{:<22}".format(*r))

print("="*150)

LLM INFERENCE OPTIMIZATION COMPARISON
Metric            Standard      MQA           GQA           Flash         Ring          RoPE                  
------------------------------------------------------------------------------------------------------------------------------------------------------
Parameters        1.05M         0.59M         0.72M         1.05M         1.05M         1.05M                 
KV Cache          Large         Small         Medium        Large         Distributed   Large                 
Attention Matrix  Stored        Stored        Stored        Not Stored    Distributed   Stored                
Memory            Highest       Lowest        Medium        Very Low      Distributed   Same                  
Inference         Slow          Fastest       Fast          Very Fast     Scalable      Same                  
Context Length    Limited       Limited       Limited       Limited       Millions      Much Longer           
Architecture      Original      Mo

**Demonstration of RoPE**

In [ ]:
# ============================================================
# Rotary Positional Embedding (RoPE) Demonstration
# ============================================================

import torch

# -------------------------------
# Rotate Half Function
# -------------------------------
def rotate_half(x):
    """
    Split vector into even and odd dimensions and rotate them.
    """

    x_even = x[..., ::2]
    x_odd = x[..., 1::2]

    rotated = torch.stack((-x_odd, x_even), dim=-1)

    return rotated.flatten(-2)


# -------------------------------
# Apply RoPE
# -------------------------------
def apply_rope(x):

    B, H, T, D = x.shape

    # Position IDs
    position = torch.arange(T).float()

    # Frequency values
    inv_freq = 1.0 / (10000 ** (torch.arange(0, D, 2).float() / D))

    # Compute angles
    angles = torch.outer(position, inv_freq)

    sin = torch.sin(angles)
    cos = torch.cos(angles)

    # Duplicate to full dimension
    sin = torch.repeat_interleave(sin, repeats=2, dim=-1)
    cos = torch.repeat_interleave(cos, repeats=2, dim=-1)

    sin = sin.unsqueeze(0).unsqueeze(0)
    cos = cos.unsqueeze(0).unsqueeze(0)

    # Apply rotation
    return x * cos + rotate_half(x) * sin


# ============================================================
# Create Dummy Query Tensor
# ============================================================

Q = torch.randn(2, 8, 128, 64)

Q_rope = apply_rope(Q)

# ============================================================
# Display Results
# ============================================================

print("=" * 70)
print("ROTARY POSITIONAL EMBEDDING DEMONSTRATION")
print("=" * 70)

print("Original Shape :", Q.shape)
print("RoPE Shape     :", Q_rope.shape)

print("\n")

# Compare several positions
positions = [0, 1, 2, 10, 50]

for pos in positions:

    print("=" * 70)
    print(f"TOKEN POSITION : {pos}")
    print("=" * 70)

    print("\nOriginal Vector (First 8 Values)")
    print(Q[0, 0, pos, :8])

    print("\nRoPE Vector (First 8 Values)")
    print(Q_rope[0, 0, pos, :8])

    diff = torch.abs(Q[0, 0, pos, :8] - Q_rope[0, 0, pos, :8])

    print("\nAbsolute Difference")
    print(diff)

    print("\nMaximum Difference :", diff.max().item())

    print("\n")

ROTARY POSITIONAL EMBEDDING DEMONSTRATION
Original Shape : torch.Size([2, 8, 128, 64])
RoPE Shape     : torch.Size([2, 8, 128, 64])


TOKEN POSITION : 0

Original Vector (First 8 Values)
tensor([-0.3542,  0.5849,  0.5340,  0.8959, -0.5414,  2.3393, -1.0841,  1.0656])

RoPE Vector (First 8 Values)
tensor([-0.3542,  0.5849,  0.5340,  0.8959, -0.5414,  2.3393, -1.0841,  1.0656])

Absolute Difference
tensor([0., 0., 0., 0., 0., 0., 0., 0.])

Maximum Difference : 0.0


TOKEN POSITION : 1

Original Vector (First 8 Values)
tensor([ 1.1025,  0.0629, -0.7307,  0.0727,  0.5230, -0.1423, -0.1177,  1.3467])

RoPE Vector (First 8 Values)
tensor([ 0.5427,  0.9617, -0.5843, -0.4448,  0.5183,  0.1585, -0.6586,  1.1805])

Absolute Difference
tensor([0.5598, 0.8988, 0.1464, 0.5175, 0.0047, 0.3008, 0.5409, 0.1661])

Maximum Difference : 0.8988111019134521


TOKEN POSITION : 2

Original Vector (First 8 Values)
tensor([-0.9235,  1.2422, -1.8533,  0.1551, -1.0438, -0.9931,  0.4939, -0.4304])

RoPE Vector (F

**Gradio Simulation**

In [ ]:
# ============================================================
# Rotary Positional Embedding (RoPE) Demonstration
#
# ============================================================

# pip install gradio matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

# ---------------------------------------------------------
# Apply RoPE
# ---------------------------------------------------------
def apply_rope(vector, position, base=10000):

    vector = np.array(vector)

    d = len(vector)

    rotated = vector.copy()

    explanation = ""

    for i in range(0, d, 2):

        theta = position / (base ** (i / d))

        cos = np.cos(theta)
        sin = np.sin(theta)

        x = vector[i]
        y = vector[i+1]

        rotated[i] = x*cos - y*sin
        rotated[i+1] = x*sin + y*cos

        explanation += f"""
Dimension Pair ({i}, {i+1})

Original:
x = {x:.4f}
y = {y:.4f}

Angle θ = {theta:.6f} radians

cos(θ) = {cos:.6f}
sin(θ) = {sin:.6f}

Rotated:
x' = {rotated[i]:.4f}
y' = {rotated[i+1]:.4f}

------------------------------------
"""

    return rotated, explanation


# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
def visualize(position):

    embedding = np.array([
        1.0,
        0.5,
        0.8,
        -0.4,
        0.3,
        0.9,
        -0.5,
        0.2
    ])

    rotated, explanation = apply_rope(
        embedding,
        position
    )

    fig, axes = plt.subplots(2,2,figsize=(10,8))

    # ---------------------------------------------
    # Original embedding
    # ---------------------------------------------

    axes[0,0].bar(
        np.arange(len(embedding)),
        embedding,
        color='royalblue'
    )

    axes[0,0].set_title("Original Embedding")

    # ---------------------------------------------
    # Rotated embedding
    # ---------------------------------------------

    axes[0,1].bar(
        np.arange(len(rotated)),
        rotated,
        color='crimson'
    )

    axes[0,1].set_title(f"RoPE Embedding (Position={position})")

    # ---------------------------------------------
    # First Pair Rotation
    # ---------------------------------------------

    axes[1,0].quiver(
        0,0,
        embedding[0],
        embedding[1],
        angles='xy',
        scale_units='xy',
        scale=1,
        color='blue',
        label='Original'
    )

    axes[1,0].quiver(
        0,0,
        rotated[0],
        rotated[1],
        angles='xy',
        scale_units='xy',
        scale=1,
        color='red',
        label='Rotated'
    )

    axes[1,0].legend()

    axes[1,0].set_xlim(-2,2)
    axes[1,0].set_ylim(-2,2)
    axes[1,0].grid(True)
    axes[1,0].set_aspect('equal')

    axes[1,0].set_title("Rotation of First Dimension Pair")

    # ---------------------------------------------
    # Difference
    # ---------------------------------------------

    axes[1,1].bar(
        np.arange(len(embedding)),
        rotated-embedding,
        color='green'
    )

    axes[1,1].set_title("Difference (Rotated - Original)")

    plt.tight_layout()

    return (
        fig,
        np.round(embedding,4),
        np.round(rotated,4),
        explanation
    )



# ---------------------------------------------------------
# Interface
# ---------------------------------------------------------

with gr.Blocks(title="RoPE Demonstration") as demo:

    gr.Markdown(
    """
# 🌀 Rotary Positional Embedding (RoPE)

Move the slider to change the token position.

Observe how every pair of embedding dimensions rotates by a different angle.

The embedding magnitude remains almost unchanged, but its direction changes according to the token position.
"""
    )

    position = gr.Slider(
        minimum=0,
        maximum=100,
        step=1,
        value=0,
        label="Token Position"
    )

    figure = gr.Plot()

    original = gr.Dataframe(
        headers=["Value"],
        label="Original Embedding"
    )

    rotated = gr.Dataframe(
        headers=["Value"],
        label="RoPE Embedding"
    )

    explanation = gr.Textbox(
        lines=20,
        label="Step-by-Step RoPE Computation"
    )

    position.change(
        visualize,
        inputs=position,
        outputs=[
            figure,
            original,
            rotated,
            explanation
        ]
    )

    demo.load(
        visualize,
        inputs=position,
        outputs=[
            figure,
            original,
            rotated,
            explanation
        ]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cccc12b8fa700effc5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
